# RAG on a Dutch PDF with Docling, OGX and pgvector

This notebook turns a PDF into searchable chunks in an OGX vector store and then
asks questions about it, with and without retrieval. As an example I use the
*Algemeen Reglement op de Elektrische Installaties* (AREI). I deliberately chose a
Dutch document and Dutch questions: all models involved are multilingual.

## What it does

1. **Parse the PDF with Docling.** Docling extracts the text with its layout and
   tables, page by page (pages 20–314; the front matter is skipped). OCR is off
   because the PDF already contains selectable text.
2. **Chunk along the document structure.** Docling's `HybridChunker` splits the
   document along sections and paragraphs, and keeps each chunk under 500 tokens of
   the embedding model's own tokenizer, so it fits e5's 512-token limit. Each chunk
   carries its section headings and page numbers as metadata.
3. **Embed and store.** The chunks are embedded with
   `multilingual-e5-large` (served by vLLM on CPU, via the OGX server) and inserted
   into a pgvector-backed OGX vector store with `vector_io.insert`. They are stored
   exactly as Docling made them, without OGX chunking them again.
4. **Ask questions.** The same question goes to the LLM (`qwen38-27b`) twice: once
   from its general knowledge only, and once with `file_search` over the vector store,
   so you can compare the answers and see which chunks were used.

## Good to know

- **Batches and resuming.** The PDF is processed 10 pages at a time to stay within
  the workbench's memory. Progress is saved in `arei_progress.json`, so after a crash
  or restart you can simply rerun the cells and it continues where it left off.
- **Embedding is slow on CPU.** About 1 second per chunk; the whole document takes
  roughly an hour.
- **Thinking is turned off** for the LLM
  (`extra_body={"extra_body": {"chat_template_kwargs": {"enable_thinking": False}}}`),
  which brings answers down from minutes to seconds.

In [ ]:
pip install ogx_client requests rich docling

In [ ]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.chunking import HybridChunker
#from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
#from transformers import AutoTokenizer
import torch

In [ ]:
import gc, json, logging, os, warnings

warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
logging.getLogger("httpx").setLevel(logging.WARNING)

In [ ]:
# create the vector store
from ogx_client import OgxClient

client = OgxClient(base_url="http://ogx-custom-server-service.ogx.svc.cluster.local:8321")

model_id = "vllm-inference/qwen38-27b"
embedding_model_id = "vllm-embedding/redhataimultilingual-e5-large"
models = client.models.list()
embedding_model = next(m for m in models if m.id == embedding_model_id)
embedding_dimension = int(embedding_model.custom_metadata["embedding_dimension"])

vector_store_id = None

for s in client.vector_stores.list().data:
    if s.name == "arei":
        vector_store_id = s.id
        break

if vector_store_id is None:
    vector_store = client.vector_stores.create(
        name="arei",
        extra_body={
            "embedding_model": embedding_model_id,
            "embedding_dimension": embedding_dimension,
            "provider_id": "pgvector",
        },
    )
    vector_store_id = vector_store.id

In [ ]:
PDF, FIRST_PAGE, LAST_PAGE = "AREI.pdf", 20, 314
PAGES_PER_BATCH = 10   # lower this if the workbench runs out of memory
EMBED_BATCH = 8       # chunks per embeddings call (~1 s each on CPU)
PROGRESS = "arei_progress.json"   # on the workbench storage, survives restarts

done = set(json.load(open(PROGRESS))) if os.path.exists(PROGRESS) else set()

def save_progress():
    with open(PROGRESS, "w") as f:
        json.dump(sorted(done), f)

def insert(records):
    vectors = client.embeddings.create(
        model=embedding_model_id, input=[r["content"] for r in records], dimensions=embedding_dimension,
    ).data
    client.vector_io.insert(vector_store_id=vector_store_id, chunks=[{
        **r,
        "embedding": v.embedding,
        "embedding_model": embedding_model_id,
        "embedding_dimension": embedding_dimension,
        "chunk_metadata": {"chunk_id": r["chunk_id"], "document_id": PDF, "source": PDF},
    } for r, v in zip(records, vectors)])

In [ ]:
torch.set_num_threads(2)

pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False  # Set to False if the PDF already has selectable text
pipeline_options.do_table_structure = True  # Keep enabled for accurate table extraction

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

# Count tokens with e5's own tokenizer, so chunks fit its 512-token limit
tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained("intfloat/multilingual-e5-large"),
    max_tokens=500,
)

# Use Docling's tokenizer-aware hybrid chunker
chunker = HybridChunker(max_tokens=512)


In [ ]:
for start in range(FIRST_PAGE, LAST_PAGE + 1, PAGES_PER_BATCH):
    end = min(start + PAGES_PER_BATCH - 1, LAST_PAGE)
    result = converter.convert(PDF, page_range=(start, end))
    records = []
    for i, chunk in enumerate(chunker.chunk(result.document)):
        chunk_id = f"arei_p{start:03d}_{i:03d}"   # same pages give the same ids, so reruns skip them
        if chunk_id in done:
            continue
        pages = sorted({p.page_no for item in chunk.meta.doc_items for p in item.prov})
        records.append({
            "chunk_id": chunk_id,
            "content": chunker.contextualize(chunk),
            "metadata": {"document_id": PDF,
                         "page_numbers": ",".join(map(str, pages)),
                         "headings": " > ".join(chunk.meta.headings or [])},
        })
    for b in range(0, len(records), EMBED_BATCH):
        batch = records[b:b + EMBED_BATCH]
        insert(batch)
        done.update(r["chunk_id"] for r in batch)
        save_progress()   # after each insert, so a crash loses at most one small batch
    print(f"pages {start}-{end}: {len(records)} new chunks, {len(done)} total")
    del result, records
    gc.collect()

In [ ]:
# query without a vector store
#system_instructions = "You are a precise and reliable AI assistant. Always use retrieved context, don't answer from general knowledge. If nothing relevant is found, say so clearly."
system_instructions = "You are a strictly grounded assistant. Answer the user's question using ONLY the provided [Context]. Never extrapolate, guess, or use outside world knowledge. If the context does not explicitly contain the answer, reply exactly: 'Information not found in provided documents.'"

query = "Hoeveel contactdozen mogen op een stroombaan volgens de AREI, het Algemeen Reglement op de elektrische installaties?"
#query = "Hoeveel stroomkringen mogen achter een differentieelschakelaar volgens de AREI, het Algemeen Reglement op de elektrische installaties?"


In [ ]:
# without lookup, and without thinking
response = client.responses.create(
    model=model_id,
    input=query,
    instructions=system_instructions,
    extra_body={"extra_body": {"chat_template_kwargs": {"enable_thinking": False}}},
)

from IPython.display import Markdown, display

display(Markdown(response.output_text))

In [ ]:
# Temporary: Due to earlier runs, data was spread across multiple vector stores
vector_store_ids = ["vs_1825a06e-e74a-4cbf-aa59-c36545ec79ca","vs_f796c217-2147-4787-85da-697a3acf05e6"]

In [ ]:
# with lookup, without thinking
response = client.responses.create(
    model=model_id,
    input=query,
    instructions=system_instructions,
    tools=[{"type": "file_search", "vector_store_ids": vector_store_ids, "max_num_results": 5}],
    include=["file_search_call.results"],# also return the retrieved chunks
    tool_choice="required", # always use tool
    extra_body={"extra_body": {"chat_template_kwargs": {"enable_thinking": False}}},
)

for item in response.output:
    if item.type == "file_search_call":
        print(f"Search ({item.status}): {item.queries}")
        for r in item.results or []:
            #print(f"  {r.score:.3f}  {r.filename}: {r.text[:100]!r}")
            print(f"  {r.score:.3f}  {r.filename}: {r.text}")
    elif item.type == "message":
        for part in item.content:
            for a in getattr(part, "annotations", None) or []:
                if a.type == "file_citation":
                    print(f"Cited: {a.filename}")

from IPython.display import Markdown, display

display(Markdown(response.output_text))